In [ ]:
"""
HUBBLE GRADIENT PREDICTION — Test 1B
======================================
Generate the specific H_0(z) curve the chi-field framework predicts:
  73.5 km/s/Mpc (z=0) -> 67.4 km/s/Mpc (z=1100), smooth sigmoid.

Compare against:
  - LCDM (constant H_0 = 67.4 everywhere, different H(z) evolution)
  - CTMT thawing model (H_0 = 72.82)

Output: Prediction for where Euclid Q2 (June 2026) should see deviations.

David Lowe (POF 2828) | March 2026
"""

In [ ]:
import jax.numpy as jnp
import numpy as np
from config import H0_LOCAL, H0_CMB, Z_CMB

In [ ]:
def chi_field_H0(z, h0_local=H0_LOCAL, h0_cmb=H0_CMB, z_transition=2.0, steepness=1.5):
    """
    Chi-field prediction: H_0 varies with redshift via sigmoid transition.

    The chi-field modifies the effective gravitational coupling:
      G_eff(z) = G / (1 + xi*kappa_0*chi(z)^2)

    As chi grows from early universe to present:
      - Early universe (z >> z_transition): chi ~ 0, G_eff ~ G, H_0 ~ H0_CMB
      - Late universe (z << z_transition): chi > 0, G_eff < G, H_0 ~ H0_LOCAL

    The transition is a smooth sigmoid centered at z_transition.

    Args:
        z: redshift (scalar or array)
        h0_local: H_0 at z=0 (local measurement)
        h0_cmb: H_0 at z=infinity (CMB)
        z_transition: redshift where transition occurs
        steepness: sharpness of transition
    """
    # Sigmoid: goes from 0 (z >> z_trans) to 1 (z << z_trans)
    # chi(z) grows as z decreases (universe ages)
    sigmoid = 1.0 / (1.0 + (z / z_transition)**steepness)
    return h0_cmb + (h0_local - h0_cmb) * sigmoid

In [ ]:
def lcdm_H(z, h0=H0_CMB, omega_m=0.315, omega_lambda=0.685):
    """
    Standard LCDM Hubble parameter H(z).
    H(z) = H_0 * sqrt(Omega_m * (1+z)^3 + Omega_Lambda)
    """
    return h0 * jnp.sqrt(omega_m * (1 + z)**3 + omega_lambda)

In [ ]:
def chi_field_H(z, h0_local=H0_LOCAL, h0_cmb=H0_CMB,
                z_transition=2.0, steepness=1.5,
                omega_m=0.315, omega_lambda=0.685):
    """
    Chi-field modified Hubble parameter H(z).
    Uses z-dependent H_0 in the Friedmann equation.
    """
    h0_z = chi_field_H0(z, h0_local, h0_cmb, z_transition, steepness)
    return h0_z * jnp.sqrt(omega_m * (1 + z)**3 + omega_lambda)

In [ ]:
def ctmt_H(z, h0=72.82, omega_m=0.315, omega_lambda=0.685, w0=-1.0, wa=-0.3):
    """
    CTMT-like thawing dark energy model.
    H(z) with CPL parameterization: w(z) = w0 + wa * z/(1+z)
    """
    a = 1.0 / (1.0 + z)
    # Dark energy density evolution
    de_factor = a**(-3*(1+w0+wa)) * jnp.exp(-3*wa*(1-a))
    return h0 * jnp.sqrt(omega_m * (1+z)**3 + omega_lambda * de_factor)

In [ ]:
def compute_predictions(z_range=None, params=None):
    """
    Compute H(z) for all three models over a redshift range.

    Returns dict with arrays for plotting and comparison.
    """
    if z_range is None:
        z_range = np.logspace(-2, 3.04, 500)  # z = 0.01 to 1100

    if params is None:
        params = {
            "z_transition": 2.0,
            "steepness": 1.5,
        }

    z = jnp.array(z_range)

    h_lcdm = np.array([float(lcdm_H(zi)) for zi in z])
    h_chi = np.array([float(chi_field_H(zi, z_transition=params["z_transition"],
                                         steepness=params["steepness"])) for zi in z])
    h_ctmt = np.array([float(ctmt_H(zi)) for zi in z])

    # Compute deviations
    delta_chi_lcdm = (h_chi - h_lcdm) / h_lcdm * 100  # percent deviation
    delta_ctmt_lcdm = (h_ctmt - h_lcdm) / h_lcdm * 100

    # Find where chi-field prediction diverges from LCDM by > 2%
    sig_idx = np.where(np.abs(delta_chi_lcdm) > 2.0)[0]
    if len(sig_idx) > 0:
        z_diverge_start = float(z_range[sig_idx[-1]])  # highest z with >2% deviation
        z_diverge_end = float(z_range[sig_idx[0]])      # lowest z
    else:
        z_diverge_start = None
        z_diverge_end = None

    # H0 values at key redshifts
    key_redshifts = [0.0, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 100.0, 1100.0]
    h0_at_z = {}
    for zk in key_redshifts:
        h0_at_z[f"z={zk}"] = {
            "H0_chi": round(float(chi_field_H0(zk, z_transition=params["z_transition"],
                                                steepness=params["steepness"])), 2),
            "H_lcdm": round(float(lcdm_H(zk)), 2),
            "H_chi": round(float(chi_field_H(zk, z_transition=params["z_transition"],
                                              steepness=params["steepness"])), 2),
        }

    return {
        "z": z_range.tolist(),
        "H_lcdm": h_lcdm.tolist(),
        "H_chi": h_chi.tolist(),
        "H_ctmt": h_ctmt.tolist(),
        "delta_chi_lcdm_pct": delta_chi_lcdm.tolist(),
        "delta_ctmt_lcdm_pct": delta_ctmt_lcdm.tolist(),
        "z_diverge_2pct_range": [z_diverge_start, z_diverge_end],
        "H0_at_key_redshifts": h0_at_z,
        "params": params,
        "euclid_prediction": {
            "note": "Euclid Q2 (June 2026) should see deviations at z < 2.0",
            "max_deviation_pct": round(float(np.max(np.abs(delta_chi_lcdm))), 2),
            "z_of_max_deviation": round(float(z_range[np.argmax(np.abs(delta_chi_lcdm))]), 4),
        }
    }

In [ ]:
def scan_transition_params(z_transitions, steepnesses):
    """
    Scan transition parameters to find best fit region.
    Returns grid of max deviations.
    """
    results = []
    for zt in z_transitions:
        for s in steepnesses:
            preds = compute_predictions(params={"z_transition": zt, "steepness": s})
            results.append({
                "z_transition": float(zt),
                "steepness": float(s),
                "max_deviation_pct": preds["euclid_prediction"]["max_deviation_pct"],
                "z_of_max_dev": preds["euclid_prediction"]["z_of_max_deviation"],
            })
    return results